In [9]:
%load_ext autoreload
%autoreload 2
%config Completer.use_jedi = False
import sys

print(sys.executable)  #
from pprint import pprint
import polars as pl
from pathlib import Path

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/home/nasim/Desktop/work/rag/sec-10k-rag/.venv/bin/python


In [10]:
from groq import Groq

In [11]:
from sec_10k_rag.common.common import write_to_parquet_file, load_parquet

In [ ]:
from sec_10k_rag.step_07_search_quality.config import get_config, ConfigType, get_llm_api_key, ROOT_DIR

In [37]:
from sec_10k_rag.step_07_search_quality.lib import generate_questions, load_chunks_text_from_db, sample_chunks, get_question_vectors, find_nearest_neighbors

In [14]:
print(dir(ConfigType)) 

['GENERATE_QUESTIONS', 'SCORE_SEARCH', '__add__', '__class__', '__contains__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getnewargs__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__members__', '__mod__', '__module__', '__mul__', '__name__', '__ne__', '__new__', '__qualname__', '__reduce__', '__reduce_ex__', '__repr__', '__rmod__', '__rmul__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', 'capitalize', 'casefold', 'center', 'count', 'encode', 'endswith', 'expandtabs', 'find', 'format', 'format_map', 'index', 'isalnum', 'isalpha', 'isascii', 'isdecimal', 'isdigit', 'isidentifier', 'islower', 'isnumeric', 'isprintable', 'isspace', 'istitle', 'isupper', 'join', 'ljust', 'lower', 'lstrip', 'maketrans', 'partition', 'removeprefix', 'removesuffix', 'replace', 'rfind', 'rindex', 'rjust', 'rpartition', 'rsplit', 'rstrip', 'split', 'splitlines',

In [15]:
config = get_config(ConfigType.GENERATE_QUESTIONS)

In [16]:
print(config)

Config(command='run', root_dir=PosixPath('/home/nasim/Desktop/work/rag/sec-10k-rag'), input_data_dir=PosixPath('/home/nasim/Desktop/work/rag/sec-10k-rag/data/processed/search_quality'), processed_data_dir=PosixPath('/home/nasim/Desktop/work/rag/sec-10k-rag/data/processed/search_quality'), log_dir=PosixPath('/home/nasim/Desktop/work/rag/sec-10k-rag/logs'), temp_dir=PosixPath('/home/nasim/Desktop/work/rag/sec-10k-rag/temp'), instructions=GenerateQuestionsInstructions(db=DbConnInfo(port='5432', dbname='sec10k', user='rag', host='127.0.0.1'), llm_info=LlmInfo(model_name='openai/gpt-oss-120b', temperature=0.0), sample_size=50, seed=42, output_file_name='questions.parquet', chunks_table='chunks', company_names={'MSFT': 'Microsoft', 'NVDA': 'Nvidia', 'TSLA': 'Tesla'}))


In [17]:
instructions = config.instructions

In [18]:
chunks_df = load_chunks_text_from_db(instructions.db, instructions.chunks_table)

In [19]:
display(chunks_df.head(10))

section_id,chunk_index,ticker,fiscal_year,item,title,text
str,i64,str,i64,str,str,str
"""MSFT-2024-item1""",0,"""MSFT""",2024,"""1""","""Business""","""Item 1 Note About Forward-Look…"
"""MSFT-2024-item1""",1,"""MSFT""",2024,"""1""","""Business""","""PART I ITEM 1. BUSINESS GENERA…"
"""MSFT-2024-item1""",2,"""MSFT""",2024,"""1""","""Business""","""We have entered a new age of A…"
"""MSFT-2024-item1""",3,"""MSFT""",2024,"""1""","""Business""","""What We Offer Founded in 1975,…"
"""MSFT-2024-item1""",4,"""MSFT""",2024,"""1""","""Business""","""PART I Item 1 The Ambitions Th…"
"""MSFT-2024-item1""",5,"""MSFT""",2024,"""1""","""Business""","""Together, the Microsoft Cloud,…"
"""MSFT-2024-item1""",6,"""MSFT""",2024,"""1""","""Business""","""Build the Intelligent Cloud an…"
"""MSFT-2024-item1""",7,"""MSFT""",2024,"""1""","""Business""","""Our cloud business benefits fr…"
"""MSFT-2024-item1""",8,"""MSFT""",2024,"""1""","""Business""","""PART I Item 1 Our AI platform,…"


In [20]:
sample_df = sample_chunks(chunks_df, n=instructions.sample_size, seed=instructions.seed)

In [21]:
print(sample_df.height)

50


In [22]:
print(chunks_df.height)

2977


In [ ]:
pprint(sorted(m.id for m in Groq(api_key=get_llm_api_key()).models.list().data))

['allam-2-7b',
 'canopylabs/orpheus-arabic-saudi',
 'canopylabs/orpheus-v1-english',
 'meta-llama/llama-prompt-guard-2-22m',
 'meta-llama/llama-prompt-guard-2-86m',
 'openai/gpt-oss-120b',
 'openai/gpt-oss-20b',
 'openai/gpt-oss-safeguard-20b',
 'qwen/qwen3.8-27b',
 'whisper-large-v3',
 'whisper-large-v3-turbo']


In [104]:
questions = generate_questions(
    sample=sample_df,
    llm_info=instructions.llm_info,
    company_names=instructions.company_names
)

In [105]:
display(questions)

section_id,chunk_index,ticker,fiscal_year,item,question,answer,quote
str,i64,str,i64,str,str,str,str
"""MSFT-2024-item7""",4,"""MSFT""",2024,"""7""","""What potential supply issue co…","""Extended disruptions at certai…","""Extended disruptions at these …"
"""NVDA-2024-item1A""",6,"""NVDA""",2024,"""1A""","""What new cloud service did Nvi…","""NVIDIA DGX Cloud services""","""We have begun offering enterpr…"
"""TSLA-2023-item7""",1,"""TSLA""",2023,"""7""","""How many consumer vehicles did…","""Tesla produced 1,845,985 consu…","""In 2023, we produced 1,845,985…"
"""MSFT-2024-item1A""",7,"""MSFT""",2024,"""1A""","""What execution and competitive…","""Microsoft says its focus on cl…","""Our focus on cloud-based and A…"
"""NVDA-2024-item15""",45,"""NVDA""",2024,"""15""","""What was the weighted average …","""Nvidia's operating leases had …","""weighted average remaining lea…"
…,…,…,…,…,…,…,…
"""MSFT-2024-item1""",10,"""MSFT""",2024,"""1""","""What AI cybersecurity product …","""Microsoft's FY 2024 AI cyberse…","""Copilot for Security, Microsof…"
"""NVDA-2024-item1""",10,"""NVDA""",2024,"""1""","""What new component did Nvidia …","""CPUs""","""CPUs in fiscal year 2024."""
"""TSLA-2025-item9A""",3,"""TSLA""",2025,"""9A""","""Did Tesla report any material …","""No, there was no change in our…","""There was no change in our int…"


In [24]:
questions_path = (
    Path(config.processed_data_dir) / instructions.output_file_name
)

In [25]:
print(questions_path)

/home/nasim/Desktop/work/rag/sec-10k-rag/data/processed/search_quality/questions.parquet


In [109]:
# write_to_parquet_file(questions, questions_path)

Writing facts to /home/nasim/Desktop/work/rag/sec-10k-rag/data/processed/search_quality/questions.parquet...
wrote 44 rows to /home/nasim/Desktop/work/rag/sec-10k-rag/data/processed/search_quality/questions.parquet in 0.02s


In [26]:
loaded_questions_df = load_parquet(questions_path)

In [27]:
print(loaded_questions_df.height)

44


In [35]:
questions_vector_df = get_question_vectors(questions_path)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [36]:
display(questions_vector_df.head(10))

question_id,section_id,chunk_index,ticker,fiscal_year,item,question,answer,quote,query,embedding
u32,str,i64,str,i64,str,str,str,str,str,"array[f32, 384]"
0,"""MSFT-2024-item7""",4,"""MSFT""",2024,"""7""","""What potential supply issue co…","""Extended disruptions at certai…","""Extended disruptions at these …","""Represent this sentence for se…","[-0.002922, 0.016542, … 0.026022]"
1,"""NVDA-2024-item1A""",6,"""NVDA""",2024,"""1A""","""What new cloud service did Nvi…","""NVIDIA DGX Cloud services""","""We have begun offering enterpr…","""Represent this sentence for se…","[-0.018953, -0.054395, … 0.041452]"
2,"""TSLA-2023-item7""",1,"""TSLA""",2023,"""7""","""How many consumer vehicles did…","""Tesla produced 1,845,985 consu…","""In 2023, we produced 1,845,985…","""Represent this sentence for se…","[0.058794, -0.008197, … 0.040833]"
3,"""MSFT-2024-item1A""",7,"""MSFT""",2024,"""1A""","""What execution and competitive…","""Microsoft says its focus on cl…","""Our focus on cloud-based and A…","""Represent this sentence for se…","[-0.006168, -0.068961, … 0.009948]"
4,"""NVDA-2024-item15""",45,"""NVDA""",2024,"""15""","""What was the weighted average …","""Nvidia's operating leases had …","""weighted average remaining lea…","""Represent this sentence for se…","[0.017486, -0.016845, … 0.034736]"
5,"""TSLA-2023-item1A""",14,"""TSLA""",2023,"""1A""","""What competitive risk does Tes…","""Tesla says it faces strong com…","""We face strong competition for…","""Represent this sentence for se…","[0.012375, -0.017351, … 0.02018]"
6,"""MSFT-2025-item1""",30,"""MSFT""",2025,"""1""","""What supply‑chain challenge co…","""Microsoft has few qualified su…","""There are few qualified suppli…","""Represent this sentence for se…","[-0.040835, -0.031272, … 0.028715]"
7,"""NVDA-2025-item1""",51,"""NVDA""",2025,"""1""","""In what year did Debora Shoqui…","""2007""","""Debora Shoquist joined NVIDIA …","""Represent this sentence for se…","[-0.057661, 0.032401, … 0.012423]"
8,"""TSLA-2023-item8""",149,"""TSLA""",2023,"""8""","""What agreement did Tesla's CEO…","""He entered into an indemnifica…","""our CEO entered into an indemn…","""Represent this sentence for se…","[-0.068122, -0.00285, … 0.034664]"


In [39]:
pprint(config)

Config(command='run',
       root_dir=PosixPath('/home/nasim/Desktop/work/rag/sec-10k-rag'),
       input_data_dir=PosixPath('/home/nasim/Desktop/work/rag/sec-10k-rag/data/processed/search_quality'),
       processed_data_dir=PosixPath('/home/nasim/Desktop/work/rag/sec-10k-rag/data/processed/search_quality'),
       log_dir=PosixPath('/home/nasim/Desktop/work/rag/sec-10k-rag/logs'),
       temp_dir=PosixPath('/home/nasim/Desktop/work/rag/sec-10k-rag/temp'),
       instructions=GenerateQuestionsInstructions(db=DbConnInfo(port='5432',
                                                                dbname='sec10k',
                                                                user='rag',
                                                                host='127.0.0.1'),
                                                  llm_info=LlmInfo(model_name='openai/gpt-oss-120b',
                                                                   temperature=0.0),
                                   

In [43]:
neighbors_df = find_nearest_neighbors(questions_vector_df, instructions.db, instructions.chunks_table, top_k=5)

In [47]:
with pl.Config(tbl_rows=-1): display(neighbors_df)

rank,section_id,chunk_index,question_id
u32,str,i64,u32
1,"""MSFT-2025-item1A""",28,0
2,"""MSFT-2024-item1A""",29,0
3,"""NVDA-2024-item1A""",12,0
4,"""TSLA-2023-item1A""",3,0
5,"""MSFT-2026-item1A""",36,0
1,"""NVDA-2025-item1A""",6,1
2,"""NVDA-2024-item1A""",6,1
3,"""NVDA-2026-item1A""",7,1
4,"""NVDA-2026-item1""",11,1
